In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path.cwd()

if (cwd / "requirements.txt").exists():
    ROOT = cwd
elif (cwd.parent / "requirements.txt").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Не найден корень проекта")

PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

predictions = pd.read_csv(
    PROCESSED / "hourly_test_predictions_extended.csv"
)

predictions["timestamp"] = pd.to_datetime(
    predictions["timestamp"], utc=True
)

predictions = predictions.set_index("timestamp").sort_index()

assert predictions.index.is_unique

grid = pd.date_range(
    "2020-03-01",
    "2020-04-01",
    freq="h",
    inclusive="left",
    tz="UTC",
    name="timestamp",
)

assert predictions.index.isin(grid).all()

calendar_predictions = predictions.reindex(grid)

print("Часов в календарной сетке:", len(grid))
print("Часов с прогнозами:", len(predictions))
print("Исключённых часов:", len(grid) - len(predictions))

Часов в календарной сетке: 744
Часов с прогнозами: 740
Исключённых часов: 4


In [2]:
def block_bootstrap_comparison(
    frame,
    candidate,
    benchmark,
    block_hours,
    repeats=5000,
    seed=42,
):
    target = frame["rv_next_1h_pct"].to_numpy(dtype=float)

    candidate_error = (
        frame[candidate].to_numpy(dtype=float) - target
    )

    benchmark_error = (
        frame[benchmark].to_numpy(dtype=float) - target
    )

    candidate_loss = candidate_error ** 2
    benchmark_loss = benchmark_error ** 2

    valid = (
        np.isfinite(candidate_loss)
        & np.isfinite(benchmark_loss)
    )

    if not valid.any():
        raise ValueError("Нет сопоставимых наблюдений")

    n = len(frame)
    n_blocks = int(np.ceil(n / block_hours))
    offsets = np.arange(block_hours)

    rng = np.random.default_rng(seed)

    sampled_delta = []
    sampled_gain = []

    for _ in range(repeats):
        starts = rng.integers(0, n, size=n_blocks)

        indices = (
            (starts[:, None] + offsets[None, :]) % n
        ).ravel()[:n]

        indices = indices[valid[indices]]

        if len(indices) == 0:
            continue

        mse_candidate = candidate_loss[indices].mean()
        mse_benchmark = benchmark_loss[indices].mean()

        sampled_delta.append(mse_candidate - mse_benchmark)

        sampled_gain.append(
            100 * (1 - mse_candidate / mse_benchmark)
        )

    actual_candidate = candidate_loss[valid].mean()
    actual_benchmark = benchmark_loss[valid].mean()

    delta_ci = np.quantile(sampled_delta, [0.025, 0.975])
    gain_ci = np.quantile(sampled_gain, [0.025, 0.975])

    return {
        "candidate": candidate,
        "benchmark": benchmark,
        "block_hours": block_hours,
        "n_observations": int(valid.sum()),
        "bootstrap_repeats": len(sampled_delta),
        "delta_mse": actual_candidate - actual_benchmark,
        "delta_ci_low": delta_ci[0],
        "delta_ci_high": delta_ci[1],
        "mse_gain_pct": 100 * (
            1 - actual_candidate / actual_benchmark
        ),
        "gain_ci_low_pct": gain_ci[0],
        "gain_ci_high_pct": gain_ci[1],
    }

In [3]:
PAIRS = [
    ("Controls_TR", "Controls"),
    ("Controls_U2", "Controls"),
    ("Controls_EX", "Controls"),
    ("Controls_TR_EX", "Controls_TR"),
    ("Controls_TR_EX", "Persistence"),
]

BLOCK_LENGTHS = [12, 24, 48]

rows = []

for block_hours in BLOCK_LENGTHS:
    for candidate, benchmark in PAIRS:
        rows.append(
            block_bootstrap_comparison(
                frame=calendar_predictions,
                candidate=candidate,
                benchmark=benchmark,
                block_hours=block_hours,
                repeats=5000,
                seed=42,
            )
        )

uncertainty = pd.DataFrame(rows)

print(
    uncertainty[
        [
            "candidate",
            "benchmark",
            "block_hours",
            "mse_gain_pct",
            "gain_ci_low_pct",
            "gain_ci_high_pct",
        ]
    ].round(4).to_string(index=False)
)

uncertainty.to_csv(
    TABLES / "forecast_block_bootstrap.csv",
    index=False,
)

     candidate   benchmark  block_hours  mse_gain_pct  gain_ci_low_pct  gain_ci_high_pct
   Controls_TR    Controls           12        0.7131           0.3602            1.5662
   Controls_U2    Controls           12        1.7678           1.1159            3.8498
   Controls_EX    Controls           12       -0.0162          -0.4522            1.4818
Controls_TR_EX Controls_TR           12        0.4284          -0.6653            4.0310
Controls_TR_EX Persistence           12      -23.8474         -63.8281           -8.2664
   Controls_TR    Controls           24        0.7131           0.3619            1.7424
   Controls_U2    Controls           24        1.7678           1.0366            4.1220
   Controls_EX    Controls           24       -0.0162          -0.5169            1.7416
Controls_TR_EX Controls_TR           24        0.4284          -0.7979            4.5617
Controls_TR_EX Persistence           24      -23.8474         -66.0092           -8.2488
   Controls_TR    Con